In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import matplotlib
from numpy import random
import copy
import scipy
import os
import sys

In [ ]:
# for the font of the plots
matplotlib.rc("font", **{"family":"serif","serif":["Computer Modern Roman"]})
matplotlib.rc("text", usetex=True)

# load data

In [ ]:
# simulation parameters
# starfish model

numx = 30
numy = 30
N = numx*numy
R = 0.5
nrec = 400
n_j = int(nrec*0.8)
wnum = int(n_j/2)+1

spacing = 1.2
dx = 0.3*spacing
dy = 0.3*spacing
Lx = spacing * 2 * R * numx
Ly = 0.5 * np.sqrt(3) * numy * spacing * 2 * R
xnum = round(Lx/dx)
ynum = round(Ly/dy)
dkx = 2*np.pi/Lx
dky = 2*np.pi/Ly
#dt = 10.0
#dt = 2.0
dt = 0.1
dw = 2*np.pi/(dt*n_j)

# frequency domain 
kxrange = np.zeros(xnum)
kyrange = np.zeros(ynum)
wrange = dw*np.arange(wnum)

for i in range(xnum):
    kxrange[i] = -np.pi/dx + dkx*i
for j in range(ynum):
    kyrange[j] = -np.pi/dy + dky*j
    
kx,ky = np.meshgrid(kxrange,kyrange,indexing='ij')


In [ ]:
# load current correlation files
# starfish model

base_dir = 'directory_of_the_data'

file_name = 'Cll_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Cll = np.fromfile(f,np.float64)
Cll = (1/N)*Cll.reshape((xnum,ynum,wnum))

file_name = 'Cltr_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Cltr = np.fromfile(f,np.float64)
Cltr = (1/N)*Cltr.reshape((xnum,ynum,wnum))

file_name = 'Clti_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Clti = np.fromfile(f,np.float64)
Clti = (1/N)*Clti.reshape((xnum,ynum,wnum))

file_name = 'Ctt_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Ctt = np.fromfile(f,np.float64)
Ctt = (1/N)*Ctt.reshape((xnum,ynum,wnum))


# get dispersion

In [ ]:
# 2d matrix whose elements are the max value of C at particular (qx,qy)
# preserve the sign of Cltr and Clti

Cll_copy = copy.copy(Cll)
Cltr_copy = copy.copy(Cltr)
Clti_copy = copy.copy(Clti)
Ctt_copy = copy.copy(Ctt)
Cfull_copy = copy.copy(Cll) + copy.copy(Ctt)

Cll_peak = np.zeros([xnum,ynum])
Cltr_peak = np.zeros([xnum,ynum])
Clti_peak = np.zeros([xnum,ynum])
Ctt_peak = np.zeros([xnum,ynum])

Cfull_peak = np.zeros([xnum,ynum])

wll_peak = np.zeros([xnum,ynum])
wltr_peak = np.zeros([xnum,ynum])
wlti_peak = np.zeros([xnum,ynum])
wtt_peak = np.zeros([xnum,ynum])

wfull_peak = np.zeros([xnum,ynum])

fromhere = 1

for i in range(xnum):
    for j in range(ynum):
        Cll_thisq = Cll_copy[i,j,fromhere:]
        Ctt_thisq = Ctt_copy[i,j,fromhere:]
        Cltr_thisq = Cltr_copy[i,j,fromhere:]
        Clti_thisq = Clti_copy[i,j,fromhere:]
        Cltr_thisq_pos = abs(Cltr_copy[i,j,fromhere:])
        Clti_thisq_pos = abs(Clti_copy[i,j,fromhere:])
        Cfull_thisq = Cfull_copy[i,j,fromhere:]
        
        Cll_peak[i,j] = np.max(Cll_thisq) # record max C value at certain q
        idx_wllmax = np.where(Cll_thisq==np.max(Cll_thisq)) # index in w dimension for the max C
        idx_wllmax = int(idx_wllmax[0][0]) + fromhere # make the index an integer and add fromhere
        wll_peak[i,j] = wrange[idx_wllmax] # record the value of max peak w. 
        
        Ctt_peak[i,j] = np.max(Ctt_thisq) # record max C value at certain q
        idx_wttmax = np.where(Ctt_thisq==np.max(Ctt_thisq)) # index in w dimension for the max C
        idx_wttmax = int(idx_wttmax[0][0]) + fromhere # make the index an integer and add fromhere
        wtt_peak[i,j] = wrange[idx_wttmax] # record the value of max peak w. 
        
        idx_wltrmax = np.where(Cltr_thisq_pos==np.max(Cltr_thisq_pos))
        idx_wltrmax = int(idx_wltrmax[0][0]) + fromhere
        wltr_peak[i,j] = wrange[idx_wltrmax]
        Cltr_peak[i,j] = Cltr_thisq[idx_wltrmax - fromhere]
        
        idx_wltimax = np.where(Clti_thisq_pos==np.max(Clti_thisq_pos))
        idx_wltimax = int(idx_wltimax[0][0]) + fromhere
        wlti_peak[i,j] = wrange[idx_wltimax]
        Clti_peak[i,j] = Clti_thisq[idx_wltimax - fromhere]
        
        Cfull_peak[i,j] = np.max(Cfull_thisq)
        idx_wfullmax = np.where(Cfull_thisq==np.max(Cfull_thisq))
        idx_wfullmax = int(idx_wfullmax[0][0]) + fromhere
        wfull_peak[i,j] = wrange[idx_wfullmax]
        

In [ ]:
C_plot = copy.copy(Cfull_peak)
w_plot = copy.copy(wfull_peak)

peak_thres = 8

kx_thres = kx[np.abs(C_plot)>peak_thres]
ky_thres = ky[np.abs(C_plot)>peak_thres]
wpk_thres = w_plot[np.abs(C_plot)>peak_thres]

#%matplotlib notebook
%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(12,10))
ax = fig.add_subplot(111,projection='3d')

img = ax.scatter3D(kx_thres,ky_thres,wpk_thres,c=C_plot[C_plot>peak_thres],cmap = 'viridis',rasterized=True)
cb = ax.figure.colorbar(img,shrink=0.7,pad=0.1)

ax.set_xlabel('$q_x$', fontsize=20, labelpad=10)
ax.set_ylabel('$q_y$', fontsize=20, labelpad=10)
ax.set_zlabel('$\omega$', fontsize=20, labelpad=10)

cb.set_label('C',fontsize=20)
ax.tick_params(labelsize=15)
cb.ax.tick_params(labelsize=15)
plt.show()


In [ ]:
# points for reciprocal lattice and the first BZ

# simulation
p1x = 0.0 # kx value of the vertex of reciprocal lattice obtained from previous result
p1y = 4*np.pi/(np.sqrt(3)*spacing)

p2x = (1/2)*p1x-(np.sqrt(3)/2)*p1y
p2y = (np.sqrt(3)/2)*p1x+(1/2)*p1y

p3x = (1/2)*p2x-(np.sqrt(3)/2)*p2y
p3y = (np.sqrt(3)/2)*p2x+(1/2)*p2y

p4x = (1/2)*p3x-(np.sqrt(3)/2)*p3y
p4y = (np.sqrt(3)/2)*p3x+(1/2)*p3y

p5x = (1/2)*p4x-(np.sqrt(3)/2)*p4y
p5y = (np.sqrt(3)/2)*p4x+(1/2)*p4y

p6x = (1/2)*p5x-(np.sqrt(3)/2)*p5y
p6y = (np.sqrt(3)/2)*p5x+(1/2)*p5y

bz1x = (1/2)*p1x + (1/(2*np.sqrt(3)))*p1y
bz1y = -(1/(2*np.sqrt(3)))*p1x + (1/2)*p1y

bz2x = (1/2)*bz1x-(np.sqrt(3)/2)*bz1y
bz2y = (np.sqrt(3)/2)*bz1x+(1/2)*bz1y

bz3x = (1/2)*bz2x-(np.sqrt(3)/2)*bz2y
bz3y = (np.sqrt(3)/2)*bz2x+(1/2)*bz2y

bz4x = (1/2)*bz3x-(np.sqrt(3)/2)*bz3y
bz4y = (np.sqrt(3)/2)*bz3x+(1/2)*bz3y

bz5x = (1/2)*bz4x-(np.sqrt(3)/2)*bz4y
bz5y = (np.sqrt(3)/2)*bz4x+(1/2)*bz4y

bz6x = (1/2)*bz5x-(np.sqrt(3)/2)*bz5y
bz6y = (np.sqrt(3)/2)*bz5x+(1/2)*bz5y


In [ ]:
C_plot = copy.copy(Cfull_peak)

%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(12,10))

img = plt.contourf(kx,ky,C_plot,levels=100)

plt.plot([p1x,p2x,p3x,p4x,p5x,p6x,p1x],[p1y,p2y,p3y,p4y,p5y,p6y,p1y],'k')
plt.plot([bz1x,bz2x,bz3x,bz4x,bz5x,bz6x,bz1x],[bz1y,bz2y,bz3y,bz4y,bz5y,bz6y,bz1y],'k')

cb = fig.colorbar(img)
plt.xlabel('$q_x$',fontsize=20)
plt.ylabel('$q_y$',fontsize=20)
cb.set_label('C',fontsize=20)
plt.gca().set_aspect('equal')
plt.show()

# path in the first Brillouin zone

In [ ]:
Gpoint = np.array([50,43],dtype=int)
Kpoint = np.array([60,58],dtype=int)
Mpoint = np.array([65,51],dtype=int)

%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(8,8))

plt.plot([kx[Gpoint[0],Gpoint[1]]],[ky[Gpoint[0],Gpoint[1]]],'ro')
plt.plot([kx[Kpoint[0],Kpoint[1]]],[ky[Kpoint[0],Kpoint[1]]],'bo')
plt.plot([kx[Mpoint[0],Mpoint[1]]],[ky[Mpoint[0],Mpoint[1]]],'go')

plt.plot([p1x,p2x,p3x,p4x,p5x,p6x,p1x],[p1y,p2y,p3y,p4y,p5y,p6y,p1y],'k')
plt.plot([bz1x,bz2x,bz3x,bz4x,bz5x,bz6x,bz1x],[bz1y,bz2y,bz3y,bz4y,bz5y,bz6y,bz1y],'k')

plt.xlabel('$q_x$',fontsize=20)
plt.ylabel('$q_y$',fontsize=20)
plt.gca().set_aspect('equal')
plt.show()

In [ ]:
# find the path Gamma -> K -> M -> Gamma
# indices for Gamma, K, M points in the first BZ

Gpoint = np.array([50,43],dtype=int)
Kpoint = np.array([60,58],dtype=int)
Mpoint = np.array([65,51],dtype=int)

# indices for the path G->K->M->G

idisGK = Kpoint[0]-Gpoint[0]
jdisGK = Kpoint[1]-Gpoint[1]
numGtoK = max(np.abs(idisGK),np.abs(jdisGK))
GtoK = np.zeros([2,numGtoK],dtype = int)
GtoK[0,0]=Gpoint[0]
GtoK[1,0]=Gpoint[1]
if np.abs(idisGK)>np.abs(jdisGK):
    idisGKsign = idisGK/np.abs(idisGK)
    for i in range(1,numGtoK):
        GtoK[0,i]=GtoK[0,0]+i*idisGKsign
        GtoK[1,i]=int(GtoK[1,0]+i*idisGKsign*jdisGK/idisGK)
else:
    jdisGKsign = jdisGK/np.abs(jdisGK)
    for i in range(1,numGtoK):
        GtoK[1,i]=GtoK[1,0]+i*jdisGKsign
        GtoK[0,i]=int(GtoK[0,0]+i*jdisGKsign*idisGK/jdisGK)

idisKM = Mpoint[0]-Kpoint[0]
jdisKM = Mpoint[1]-Kpoint[1]
numKtoM = max(np.abs(idisKM),np.abs(jdisKM))
KtoM = np.zeros([2,numKtoM],dtype = int)
KtoM[0,0]=Kpoint[0]
KtoM[1,0]=Kpoint[1]
if np.abs(idisKM)>np.abs(jdisKM):
    idisKMsign = idisKM/np.abs(idisKM)
    for i in range(1,numKtoM):
        KtoM[0,i]=KtoM[0,0]+i*idisKMsign
        KtoM[1,i]=int(KtoM[1,0]+i*idisKMsign*jdisKM/idisKM)
else:
    jdisKMsign = jdisKM/np.abs(jdisKM)
    for i in range(1,numKtoM):
        KtoM[1,i]=KtoM[1,0]+i*jdisKMsign
        KtoM[0,i]=int(KtoM[0,0]+i*jdisKMsign*idisKM/jdisKM)
        
idisMG = Gpoint[0]-Mpoint[0]
jdisMG = Gpoint[1]-Mpoint[1]
numMtoG = max(np.abs(idisMG),np.abs(jdisMG))
MtoG = np.zeros([2,numMtoG+1],dtype = int)
MtoG[0,0]=Mpoint[0]
MtoG[1,0]=Mpoint[1]
if np.abs(idisMG)>np.abs(jdisMG):
    idisMGsign = idisMG/np.abs(idisMG)
    for i in range(1,numMtoG):
        MtoG[0,i]=MtoG[0,0]+i*idisMGsign
        MtoG[1,i]=int(MtoG[1,0]+i*idisMGsign*jdisMG/idisMG)
else:
    jdisMGsign = jdisMG/np.abs(jdisMG)
    for i in range(1,numMtoG):
        MtoG[1,i]=MtoG[1,0]+i*jdisMGsign
        MtoG[0,i]=int(MtoG[0,0]+i*jdisMGsign*idisMG/jdisMG)
MtoG[0,-1]=Gpoint[0]
MtoG[1,-1]=Gpoint[1]


In [ ]:
C_plot = copy.copy(Cfull_peak)

%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(10,8))
img = plt.contourf(kx,ky,C_plot)

plt.plot([p1x,p2x,p3x,p4x,p5x,p6x,p1x],[p1y,p2y,p3y,p4y,p5y,p6y,p1y],'k')
plt.plot([bz1x,bz2x,bz3x,bz4x,bz5x,bz6x,bz1x],[bz1y,bz2y,bz3y,bz4y,bz5y,bz6y,bz1y],'k')
plt.plot([kx[Gpoint[0],Gpoint[1]],kx[Kpoint[0],Kpoint[1]],kx[Mpoint[0],Mpoint[1]]],[ky[Gpoint[0],Gpoint[1]],ky[Kpoint[0],Kpoint[1]],ky[Mpoint[0],Mpoint[1]]],'mo')
plt.plot(kx[GtoK[0,:],GtoK[1,:]],ky[GtoK[0,:],GtoK[1,:]],'ro')
plt.plot(kx[KtoM[0,:],KtoM[1,:]],ky[KtoM[0,:],KtoM[1,:]],'o',color='orange')
plt.plot(kx[MtoG[0,:],MtoG[1,:]],ky[MtoG[0,:],MtoG[1,:]],'bo')

cb = fig.colorbar(img)
plt.xlabel('$q_x$',fontsize=20)
plt.ylabel('$q_y$',fontsize=20)
cb.set_label('C',fontsize=20)
plt.gca().set_aspect('equal')
plt.show()

In [ ]:
# along the path in the first Brillouin zone Gamma -> K -> M -> Gamma
Cll_GtoK = copy.copy(Cll)
Cll_GtoK = Cll_GtoK[GtoK[0,:],GtoK[1,:],:]
Cll_KtoM = copy.copy(Cll)
Cll_KtoM = Cll_KtoM[KtoM[0,:],KtoM[1,:],:]
Cll_MtoG = copy.copy(Cll)
Cll_MtoG = Cll_MtoG[MtoG[0,:],MtoG[1,:],:]
Cll_GKMG = np.concatenate((Cll_GtoK,Cll_KtoM,Cll_MtoG),axis=0)

Cltr_GtoK = copy.copy(Cltr)
Cltr_GtoK = Cltr_GtoK[GtoK[0,:],GtoK[1,:],:]
Cltr_KtoM = copy.copy(Cltr)
Cltr_KtoM = Cltr_KtoM[KtoM[0,:],KtoM[1,:],:]
Cltr_MtoG = copy.copy(Cltr)
Cltr_MtoG = Cltr_MtoG[MtoG[0,:],MtoG[1,:],:]
Cltr_GKMG = np.concatenate((Cltr_GtoK,Cltr_KtoM,Cltr_MtoG),axis=0)

Clti_GtoK = copy.copy(Clti)
Clti_GtoK = Clti_GtoK[GtoK[0,:],GtoK[1,:],:]
Clti_KtoM = copy.copy(Clti)
Clti_KtoM = Clti_KtoM[KtoM[0,:],KtoM[1,:],:]
Clti_MtoG = copy.copy(Clti)
Clti_MtoG = Clti_MtoG[MtoG[0,:],MtoG[1,:],:]
Clti_GKMG = np.concatenate((Clti_GtoK,Clti_KtoM,Clti_MtoG),axis=0)

Ctt_GtoK = copy.copy(Ctt)
Ctt_GtoK = Ctt_GtoK[GtoK[0,:],GtoK[1,:],:]
Ctt_KtoM = copy.copy(Ctt)
Ctt_KtoM = Ctt_KtoM[KtoM[0,:],KtoM[1,:],:]
Ctt_MtoG = copy.copy(Ctt)
Ctt_MtoG = Ctt_MtoG[MtoG[0,:],MtoG[1,:],:]
Ctt_GKMG = np.concatenate((Ctt_GtoK,Ctt_KtoM,Ctt_MtoG),axis=0)

qnum = np.size(Cll_GKMG,axis=0)
q_band,w_band = np.meshgrid(np.arange(qnum),wrange,indexing='ij')

fromthisw = 1 # not include w=0
uptothisw = -1 # high frequency w is not relevant
w_band = w_band[:,fromthisw:uptothisw] # only take w>0.
q_band = q_band[:,fromthisw:uptothisw]
Cll_band = Cll_GKMG[:,fromthisw:uptothisw]
Cltr_band = Cltr_GKMG[:,fromthisw:uptothisw]
Clti_band = Clti_GKMG[:,fromthisw:uptothisw]
Ctt_band = Ctt_GKMG[:,fromthisw:uptothisw]


# dispersion

In [ ]:
# need to find the range of indices to delete the data corresponding to the self-circling, as it is different for each data

tickloc = [0,np.size(GtoK,axis=1),np.size(GtoK,axis=1)+np.size(KtoM,axis=1)-1,np.size(GtoK,axis=1)+np.size(KtoM,axis=1)+np.size(MtoG,axis=1)-1]

C_band = copy.copy(Cll_band)+copy.copy(Ctt_band)

%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(10,8))
img = plt.contourf(q_band,w_band,C_band,levels=100)

#plt.plot([0,38],[w_band[0,54],w_band[0,54]])
#plt.plot([0,38],[w_band[0,60],w_band[0,60]])
plt.plot([0,37],[w_band[0,20],w_band[0,20]])
plt.plot([0,37],[w_band[0,24],w_band[0,24]])
#plt.plot([0,37],[w_band[0,12],w_band[0,12]])
#plt.plot([0,37],[w_band[0,16],w_band[0,16]])
#plt.plot([0,37],[w_band[0,4],w_band[0,4]])
#plt.plot([0,37],[w_band[0,1],w_band[0,1]])

cb = fig.colorbar(img)
plt.xticks(tickloc,["$\Gamma$","K","M","$\Gamma$"],fontsize=20)
plt.ylabel('$\omega$',fontsize=20)
cb.set_label('C',fontsize=20)

plt.show()

In [ ]:
# exclude the self-circling signal part for better fitting

Cll_noSC = copy.copy(Cll_band)
#Cll_noSC[:,1:4]=0
#Cll_noSC[:,12:16]=0
Cll_noSC[:,20:24]=0
#Cll_noSC[:,54:60]=0
#Cll_noSC[:,110:116]=0
Ctt_noSC = copy.copy(Ctt_band)
#Ctt_noSC[:,1:4]=0
#Ctt_noSC[:,12:16]=0
Ctt_noSC[:,20:24]=0
#Ctt_noSC[:,54:60]=0
#Ctt_noSC[:,110:116]=0
Cltr_noSC = copy.copy(Cltr_band)
#Cltr_noSC[:,1:4]=0
#Cltr_noSC[:,12:16]=0
#Cltr_noSC[:,20:24]=0
#Cltr_noSC[:,54:60]=0
#Cltr_noSC[:,110:116]=0
Clti_noSC = copy.copy(Clti_band)
#Clti_noSC[:,1:4]=0
#Clti_noSC[:,12:16]=0
#Clti_noSC[:,20:24]=0
#Clti_noSC[:,54:60]=0
#Clti_noSC[:,110:116]=0

In [ ]:
# dispersion result after removing the self-circling part

tickloc = [0,np.size(GtoK,axis=1),np.size(GtoK,axis=1)+np.size(KtoM,axis=1)-1,np.size(GtoK,axis=1)+np.size(KtoM,axis=1)+np.size(MtoG,axis=1)-1]

C_band = copy.copy(Cll_noSC)+copy.copy(Ctt_noSC)

%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(10,8))
img = plt.contourf(q_band,w_band,C_band,levels=100)

cb = fig.colorbar(img)
plt.xticks(tickloc,["$\Gamma$","K","M","$\Gamma$"],fontsize=20)
plt.ylabel('$\omega$',fontsize=20)
cb.set_label('C',fontsize=20)

plt.show()

# 3d dispersion with reciprocal lattice and BZ

In [ ]:
C_plot = copy.copy(Cfull_peak)
w_plot = copy.copy(wfull_peak)

peak_thres = 30

kx_thres = kx[np.abs(C_plot)>peak_thres]
ky_thres = ky[np.abs(C_plot)>peak_thres]
wpk_thres = w_plot[np.abs(C_plot)>peak_thres]

#%matplotlib notebook
%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(12,10))
ax = fig.add_subplot(111,projection='3d')

img = ax.scatter3D(kx_thres,ky_thres,wpk_thres,c=C_plot[C_plot>peak_thres],cmap = 'viridis',rasterized=True)
ax.plot([p1x,p2x,p3x,p4x,p5x,p6x,p1x],[p1y,p2y,p3y,p4y,p5y,p6y,p1y],zs=0.03,zdir='z',color='k')
ax.plot([bz1x,bz2x,bz3x,bz4x,bz5x,bz6x,bz1x],[bz1y,bz2y,bz3y,bz4y,bz5y,bz6y,bz1y],zs=0.03,zdir='z',color='k')
ax.plot([kx[Gpoint[0],Gpoint[1]],kx[Kpoint[0],Kpoint[1]],kx[Mpoint[0],Mpoint[1]],kx[Gpoint[0],Gpoint[1]]],[ky[Gpoint[0],Gpoint[1]],ky[Kpoint[0],Kpoint[1]],ky[Mpoint[0],Mpoint[1]],ky[Gpoint[0],Gpoint[1]]],zs=0.03,zdir='z',color='r',linestyle='--')

#cb = fig.colorbar(img,extend='max')
cb = ax.figure.colorbar(img,shrink=0.7,pad=0.1)

ax.set_xlabel('$q_x$', fontsize=20, labelpad=10)
ax.set_ylabel('$q_y$', fontsize=20, labelpad=10)
ax.set_zlabel('$\omega$', fontsize=20, labelpad=10)

ax.set_zlim([0.0,0.3])

cb.set_label('C',fontsize=20)
ax.tick_params(labelsize=15)
cb.ax.tick_params(labelsize=15)
plt.show()

# plot showing all

In [ ]:
C_plot = copy.copy(Cfull_peak)
w_plot = copy.copy(wfull_peak)

peak_thres = 3

kx_thres = kx[np.abs(C_plot)>peak_thres]
ky_thres = ky[np.abs(C_plot)>peak_thres]
wpk_thres = w_plot[np.abs(C_plot)>peak_thres]

#w0 = 0.03
w0 = 0.72*2*np.pi

#%matplotlib notebook
%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(20,5))
ax1 = fig.add_subplot(1,3,1,projection='3d')
ax2 = fig.add_subplot(1,3,2)
ax3 = fig.add_subplot(1,3,3)

# first plot
img1 = ax1.scatter3D(kx_thres,ky_thres,wpk_thres,c=C_plot[C_plot>peak_thres],cmap = 'viridis',rasterized=True)
ax1.plot([p1x,p2x,p3x,p4x,p5x,p6x,p1x],[p1y,p2y,p3y,p4y,p5y,p6y,p1y],zs=w0,zdir='z',color='k')
ax1.plot([bz1x,bz2x,bz3x,bz4x,bz5x,bz6x,bz1x],[bz1y,bz2y,bz3y,bz4y,bz5y,bz6y,bz1y],zs=w0,zdir='z',color='k')
ax1.plot([kx[Gpoint[0],Gpoint[1]],kx[Kpoint[0],Kpoint[1]],kx[Mpoint[0],Mpoint[1]],kx[Gpoint[0],Gpoint[1]]],[ky[Gpoint[0],Gpoint[1]],ky[Kpoint[0],Kpoint[1]],ky[Mpoint[0],Mpoint[1]],ky[Gpoint[0],Gpoint[1]]],zs=w0,zdir='z',color='r',linestyle='--')

cb1 = ax1.figure.colorbar(img1,shrink=0.8,pad=0.12)
ax1.set_xlabel('$q_x$', fontsize=20, labelpad=10)
ax1.set_ylabel('$q_y$', fontsize=20, labelpad=10)
ax1.set_zlabel('$\omega$', fontsize=20, labelpad=10)
ax1.set_zlim([0.0,10.0])
cb1.set_label('C',fontsize=20)
ax1.tick_params(labelsize=15)
cb1.ax.tick_params(labelsize=15)
ax1.text2D(-0.06,0.09,'(a)',fontsize=30,va='top',ha='right')

# second plot
tickloc = [0,np.size(GtoK,axis=1),np.size(GtoK,axis=1)+np.size(KtoM,axis=1)-1,np.size(GtoK,axis=1)+np.size(KtoM,axis=1)+np.size(MtoG,axis=1)-1]

C_band = copy.copy(Cll_band)+copy.copy(Ctt_band)

img2 = ax2.contourf(q_band,w_band,C_band,levels=100)

cb2 = ax2.figure.colorbar(img2)
ax2.tick_params(labelsize=15)
cb2.ax.tick_params(labelsize=15)
ax2.set_xticks(tickloc,["$\Gamma$","K","M","$\Gamma$"],fontsize=20)
ax2.set_ylabel('$\omega$',fontsize=20)
cb2.set_label('C',fontsize=20)
ax2.text(-2.5,31.0,'(b)',fontsize=30,va='top',ha='right')

# third plot
CnoSC_band = copy.copy(Cll_noSC)+copy.copy(Ctt_noSC)

img3 = ax3.contourf(q_band,w_band,CnoSC_band,levels=100)

cb3 = ax3.figure.colorbar(img3)
ax3.tick_params(labelsize=15)
cb3.ax.tick_params(labelsize=15)
ax3.set_xticks(tickloc,["$\Gamma$","K","M","$\Gamma$"],fontsize=20)
ax3.set_ylabel('$\omega$',fontsize=20)
cb3.set_label('C',fontsize=20)
ax3.text(-2.5,31.0,'(c)',fontsize=30,va='top',ha='right')

# to avoid the weird white lines appearing in pdf file of the plot
for c in img2.collections:
    c.set_edgecolor("face")

for c in img3.collections:
    c.set_edgecolor("face")

fig.tight_layout()
plt.show()
#fig.savefig('disp_f006_w072twopi_ve0007sqrt20_100runs.pdf',dpi=100,bbox_inches='tight')